# Session 13 · Case study: leaderboard round L1 with a TF-IDF classifier

**Block 2 practice.** Train TF-IDF + linear classifiers, choose one on the validation years, evaluate it the way a task with more than 1,000 headings needs (baseline, macro-F1, per-heading results, top-k accuracy, time versus random split), and write the first leaderboard submission (`id,heading`).

Reference values on the public leaderboard (accuracy / macro-F1): always 3926 0.041 / 0.000 · **word TF-IDF on the 50k sample 0.806 / 0.517** · word TF-IDF on the full training set 0.872 / 0.682 · character TF-IDF on the full training set 0.882 / 0.695.

Run time: about 9 minutes for sections 1–5 on a laptop; the optional full-training-set run in section 6 takes about 10 more minutes and 8 GB of memory.

Theory: [02-tfidf-and-text-classification.md](../theory/02-tfidf-and-text-classification.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
texts = decisions["description"]                     # one document = one description of goods
print(decisions.shape, decisions["heading"].nunique(), "headings")

In [ ]:
year = decisions["start_date"].dt.year
train, valid = decisions[year <= 2021], decisions[year >= 2022]   # validation by time, as on the leaderboard
print(len(train), "training decisions (2017-2021),", len(valid), "validation decisions (2022-2023)")

## 1. Word and character models on the validation years

In [ ]:
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import make_pipeline


def svm(alpha=1e-5):
    """A linear SVM fitted by stochastic gradient descent: fast with 1,000 classes."""
    return SGDClassifier(loss="hinge", alpha=alpha, max_iter=20, tol=None, random_state=0, n_jobs=-1)


candidates = {
    "word unigrams": lambda: make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), svm()),
    "word unigrams, alpha=3e-5": lambda: make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), svm(3e-5)),
    "char_wb 3-5": lambda: make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=300_000),
        svm()),
}
rows, fitted = [], {}
for name, make in candidates.items():
    t0 = time.perf_counter()
    model = make().fit(train["description"], train["heading"])
    pred = model.predict(valid["description"])
    fitted[name] = (model, pred)
    rows.append({"model": name, "accuracy": accuracy_score(valid["heading"], pred),
                 "macro_F1": f1_score(valid["heading"], pred, average="macro"),
                 "seconds": time.perf_counter() - t0})
pd.DataFrame(rows).round(3)

**Question.** The character model is better by about 1.5 points and takes five times longer. When is that trade worth it?

## 2. A baseline, top-k accuracy and the choice of split

Three checks that a task with more than 1,000 headings needs before any model is trusted:

- the **simplest baseline** (always the most frequent heading) shows how much accuracy alone can hide;
- **top-k accuracy** measures the model as an officer would use it: is the true heading among the first k suggestions?
- a **random split** of the same size shows how much more optimistic the validation would be without the split by time.

In [ ]:
from sklearn.model_selection import train_test_split

always_3926 = np.repeat("3926", len(valid))
print("always 3926: accuracy", round(accuracy_score(valid["heading"], always_3926), 3),
      "macro-F1", round(f1_score(valid["heading"], always_3926, average="macro"), 4))

word_model, word_pred = fitted["word unigrams"]
scores = word_model.decision_function(valid["description"])       # one score per heading
order = np.argsort(-scores, axis=1)
y_val = valid["heading"].to_numpy()
top_k = {k: (word_model.classes_[order[:, :k]] == y_val[:, None]).any(axis=1).mean() for k in [1, 3, 5, 10]}
print("top-k accuracy of the word model:", {k: round(v, 3) for k, v in top_k.items()})

rand_train, rand_valid = train_test_split(decisions, test_size=len(valid), random_state=0)
rand_pred = candidates["word unigrams"]().fit(rand_train["description"], rand_train["heading"]).predict(rand_valid["description"])
pd.DataFrame({"split": ["by time (2017-2021 -> 2022-2023)", "random, same sizes"],
              "accuracy": [accuracy_score(valid["heading"], word_pred), accuracy_score(rand_valid["heading"], rand_pred)],
              "macro_F1": [f1_score(valid["heading"], word_pred, average="macro"),
                           f1_score(rand_valid["heading"], rand_pred, average="macro")]}).round(3)

**Questions.** (1) How many suggestions would an officer need to see so that the true heading is in the list for 85 % of the requests? (2) Which split predicts the leaderboard better? Compare both numbers with the leaderboard scores of section 5.

## 3. Per-heading metrics

In [ ]:
model, pred = fitted["char_wb 3-5"]
report = pd.DataFrame(classification_report(valid["heading"], pred, output_dict=True, zero_division=0)).T.iloc[:-3]
report["name"] = nomenclature["heading_description"].reindex(report.index).str[:50]
print("headings in the validation years:", len(report), "| with F1 = 0:", int((report["f1-score"] == 0).sum()))
report.sort_values("support", ascending=False).head(10).round(2)

In [ ]:
n_train = train["heading"].value_counts().reindex(report.index, fill_value=0)
bins = pd.cut(n_train, [-1, 0, 10, 50, 200, np.inf], labels=["0", "1-10", "11-50", "51-200", ">200"])
report["f1-score"].groupby(bins, observed=True).agg(["mean", "count"]).round(2)

**Question.** Which headings have high precision but low recall? Pick one and look at its English description: what kind of goods might the model miss?

## 4. Refit on all sample decisions and predict the test set

In [ ]:
test = pd.read_parquet(DATA / "test.parquet")
final = candidates["char_wb 3-5"]()
final.fit(decisions["description"], decisions["heading"])             # all 50,000 decisions, 2017-2023
submission = pd.DataFrame({"id": test["id"], "heading": final.predict(test["description"])})
print(submission.shape)
print(submission["heading"].value_counts(normalize=True).head(5).round(3))

Checks before submitting: one row per test decision, no missing ids, four-digit headings.

In [ ]:
assert len(submission) == len(test) and submission["id"].is_unique
assert submission["heading"].str.fullmatch(r"\d{4}").all()
path = Path("submission-L1-tfidf.csv")
submission.to_csv(path, index=False)
print("written:", path.resolve())

## 5. Score

Upload the file to the course leaderboard, or score it locally where the solution file is available (lecturer):

```bash
uv run python case-study/score.py sessions/13-classical-nlp/workbooks/submission-L1-tfidf.csv
```

The next cell does the same if `case-study/data/instructor/solution.csv` exists. Expected with this notebook: about 0.82 public and 0.80 private accuracy.

In [ ]:
import sys

solution = DATA / "instructor" / "solution.csv"
if solution.exists():
    sys.path.insert(0, str(ROOT / "case-study"))
    from score import score
    for k, v in score(submission, pd.read_csv(solution, dtype=str)).items():
        print(f"{k:22s} {v:.4f}")
else:
    print("No solution file: upload the CSV to the leaderboard instead.")

## 6. Optional: the full training set

The sample has on average 55 decisions per heading; the full training set has 309,529 decisions. Set `RUN_FULL = True` to train the word model on all of them (about 4 minutes; the character model needs about 10 minutes and 8 GB of memory). The reference values are 0.872 (word) and 0.882 (character) public accuracy.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    full = pd.read_parquet(DATA / "train.parquet", columns=["description", "heading"])
    big = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True, max_features=400_000), svm(2e-6))
    big.fit(full["description"], full["heading"])
    submission_full = pd.DataFrame({"id": test["id"], "heading": big.predict(test["description"])})
    submission_full.to_csv("submission-L1-tfidf-full.csv", index=False)
    print(submission_full.shape)

## Your turn

1. Why is the public leaderboard score (about 0.82) higher than the validation score (about 0.79), and the private score lower again? Write down two reasons.
2. Try one more setting on the validation years (for example `ngram_range=(2, 4)` for characters, or word unigrams plus bigrams) and keep the best. Report accuracy *and* macro-F1.
3. Record your L1 submission in the team log: date, model, settings, validation accuracy and macro-F1, leaderboard scores. Session 14 (round L2) and Session 16 (round L3) compare against it.

In [ ]:
# your code here